# Testing stanza - Stanford NLP

In [1]:
import stanza

from src.data.preprocessing import preprocessing_legal_pt_voto_relatorio

In [2]:
stanza.download('pt')

2025-10-01 11:09:40 INFO: Downloaded file to /home/trdp/stanza_resources/resources.json
2025-10-01 11:09:40 INFO: Downloading default packages for language: pt (Portuguese) ...


2025-10-01 11:10:46 INFO: Downloaded file to /home/trdp/stanza_resources/pt/default.zip
2025-10-01 11:10:48 INFO: Finished downloading models and saved to /home/trdp/stanza_resources


In [3]:
nlp = stanza.Pipeline('pt')

2025-10-01 11:10:48 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2025-10-01 11:10:49 INFO: Downloaded file to /home/trdp/stanza_resources/resources.json
2025-10-01 11:10:49 INFO: Loading these models for language: pt (Portuguese):
| Processor    | Package         |
----------------------------------
| tokenize     | bosque          |
| mwt          | bosque          |
| pos          | bosque_charlm   |
| lemma        | bosque_nocharlm |
| constituency | cintil_charlm   |
| depparse     | bosque_charlm   |

2025-10-01 11:10:49 INFO: Using device: cuda
2025-10-01 11:10:49 INFO: Loading: tokenize
2025-10-01 11:10:50 INFO: Loading: mwt
2025-10-01 11:10:50 INFO: Loading: pos
2025-10-01 11:10:52 INFO: Loading: lemma
2025-10-01 11:10:52 INFO: Loading: constituency
2025-10-01 11:10:52 INFO: Loading: depparse
2025-10-01 11:10:53 INFO: Done loading processors!


In [4]:
doc = nlp("Embora Maria estivesse trabalhando até tarde no escritório, correu para casa assim que recebeu uma ligação do irmão mais novo, que afirmava que o cachorro deles estava latindo furiosamente para algo no quintal. Quando chegou, no entanto, a única coisa que encontrou foi uma lixeira virada, o que sugeria que um guaxinim — ou talvez um gato de rua — tinha causado a confusão. Aliviada, mas ainda curiosa, Maria decidiu instalar uma pequena câmera perto da cerca para finalmente descobrir o que vinha perturbando o cachorro todas as noites.")

In [6]:
for sentence in doc.sentences:
    print("Sentence:", sentence.text)
    # Print word-by-word dependency info
    for word in sentence.words:
        print(
            f"Word: {word.text}\tHead: {sentence.words[word.head - 1].text if word.head > 0 else 'ROOT'}\tRelation: {word.deprel}"
        )

Sentence: Embora Maria estivesse trabalhando até tarde no escritório, correu para casa assim que recebeu uma ligação do irmão mais novo, que afirmava que o cachorro deles estava latindo furiosamente para algo no quintal.
Word: Embora	Head: trabalhando	Relation: mark
Word: Maria	Head: trabalhando	Relation: nsubj
Word: estivesse	Head: trabalhando	Relation: aux
Word: trabalhando	Head: correu	Relation: advcl
Word: até	Head: tarde	Relation: case
Word: tarde	Head: trabalhando	Relation: advmod
Word: em	Head: escritório	Relation: case
Word: o	Head: escritório	Relation: det
Word: escritório	Head: trabalhando	Relation: obl
Word: ,	Head: trabalhando	Relation: punct
Word: correu	Head: ROOT	Relation: root
Word: para	Head: casa	Relation: case
Word: casa	Head: correu	Relation: obl
Word: assim	Head: recebeu	Relation: advmod
Word: que	Head: recebeu	Relation: mark
Word: recebeu	Head: correu	Relation: advcl
Word: uma	Head: ligação	Relation: det
Word: ligação	Head: recebeu	Relation: obj
Word: de	Head: irm

In [7]:
from spacy import displacy
for sentence in doc.sentences:
    edges = []
    for word in sentence.words:
        if word.head > 0:  # not root
            edges.append({
                "dep": word.deprel,
                "governor": sentence.words[word.head - 1].text,
                "dependent": word.text
            })

    # Convert to displaCy format
    words = [{"text": w.text, "tag": w.upos} for w in sentence.words]
    arcs = []
    for i, w in enumerate(sentence.words):
        if w.head > 0:
            start = min(i, w.head - 1)
            end = max(i, w.head - 1)
            arcs.append({
                "start": start,
                "end": end,
                "label": w.deprel,
                "dir": "left" if i < w.head - 1 else "right"
            })

    ex = {"words": words, "arcs": arcs}
    displacy.render(ex, style="dep", manual=True)

In [3]:
import logging
import networkx as nx
from pyvis.network import Network
from typing import Dict, Tuple, List, Optional
import spacy
import pandas as pd
from collections import Counter
# -----------------------------
# Configuration / constants
# -----------------------------

ENTITY_NODE_PREFIX = "ENT"
SENT_TOKEN_NODE_FMT = "s{sent_idx}_w{word_id}"
NEXT_SENT_EDGE_LABEL = "next_sent"
MENTION_EDGE_LABEL = "mention_of"
DEP_REL_LABEL = "dep"

# For reproducibility in visual layouts
RANDOM_SEED = 42

# -----------------------------
# Setup logging
# -----------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

# -----------------------------
# NLP setup
# -----------------------------

logger.info("Loading spaCy Portuguese model pt_core_news_lg...")
nlp = spacy.load("pt_core_news_lg")

# -----------------------------
# Graph building functions
# -----------------------------

def add_sentence_dependency_subgraph(
    G: nx.MultiDiGraph,
    sentence: spacy.tokens.Span,
    sent_idx: int,
    root_nodes: List[str],
    add_reverse: bool = False,
) -> None:
    """
    Add nodes and dependency edges of a single sentence into the graph G.
    Populates root_nodes with the node ID of the root(s).
    """
    for token in sentence:
        node_id = SENT_TOKEN_NODE_FMT.format(sent_idx=sent_idx, word_id=token.i)
        G.add_node(
            node_id,
            text=token.text,
            lemma=token.lemma_,
            upos=token.pos_,
            deprel=token.dep_,
            sent_idx=sent_idx,
            token_idx=token.i,
            entity=False  # default
        )

    for token in sentence:
        child_id = SENT_TOKEN_NODE_FMT.format(sent_idx=sent_idx, word_id=token.i)
        head = token.head
        if head.i == token.i:
            # root of this sentence
            root_nodes.append(child_id)
        else:
            head_id = SENT_TOKEN_NODE_FMT.format(sent_idx=sent_idx, word_id=head.i)
            # Forward edge
            G.add_edge(
                head_id, child_id,
                label=token.dep_,
                relation_type="dependency"
            )
            if add_reverse:
                G.add_edge(
                    child_id, head_id,
                    label=token.dep_ + "_rev",
                    relation_type="dependency_rev"
                )

# def add_entity_mentions(
#     G: nx.MultiDiGraph,
#     doc: spacy.tokens.Doc
# ) -> None:
#     """
#     Add entity mention nodes and connect them to token nodes.
#     """
#     for ent in doc.ents:
#         ent_id = f"{ENTITY_NODE_PREFIX}_{ent.start_char}_{ent.end_char}"
#         logger.debug(f"Adding entity node {ent_id} for entity '{ent.text}' type={ent.label_}")
#         G.add_node(
#             ent_id,
#             text=ent.text,
#             ent_type=ent.label_,
#             entity=True
#         )
#         for tok in ent:
#             token_node_id = SENT_TOKEN_NODE_FMT.format(sent_idx=tok.sent.start, word_id=tok.i)
#             if not G.has_node(token_node_id):
#                 logger.warning(f"Token node {token_node_id} not found when linking entity {ent_id}")
#                 continue
#             G.add_edge(ent_id, token_node_id, label=MENTION_EDGE_LABEL, relation_type="mention")
#             G.add_edge(token_node_id, ent_id, label=MENTION_EDGE_LABEL + "_rev", relation_type="mention_rev")

def connect_sentence_roots(
    G: nx.MultiDiGraph,
    root_nodes: List[str]
) -> None:
    """
    Connect roots of consecutive sentences via NEXT_SENT_EDGE_LABEL edges.
    """
    for i in range(len(root_nodes) - 1):
        u = root_nodes[i]
        v = root_nodes[i + 1]
        G.add_edge(u, v, label=NEXT_SENT_EDGE_LABEL, relation_type="inter_sent")
        G.add_edge(v, u, label=NEXT_SENT_EDGE_LABEL + "_rev", relation_type="inter_sent_rev")

def build_document_graph(text: str, add_reverse: bool = False) -> nx.MultiDiGraph:
    """
    Build full document graph: dependency trees + entity mentions + sentence root connections.
    """
    doc = nlp(text)

    G = nx.MultiDiGraph()
    root_nodes: List[str] = []

    for sent_idx, sentence in enumerate(doc.sents):
        add_sentence_dependency_subgraph(G, sentence, sent_idx, root_nodes, add_reverse)

    # add_entity_mentions(G, doc)
    connect_sentence_roots(G, root_nodes)

    return G

# -----------------------------
# Visualization functions
# -----------------------------

def visualize_graph_pyvis(
    G: nx.MultiDiGraph,
    output_html: str,
    fixed_layout: bool = True,
    width: str = "100%",
    height: str = "800px"
) -> None:
    """
    Use Pyvis to produce an interactive HTML visualization.
    """
    net = Network(
        height=height, width=width, directed=True,
        notebook=False, select_menu=True, filter_menu=True,
        neighborhood_highlight=True
    )
    net.show_buttons(filter_=['physics'])

    positions: Optional[Dict[str, Tuple[float, float]]] = None
    if fixed_layout:
        positions = nx.spring_layout(G, seed=RANDOM_SEED)

    # Add nodes
    for node_id, node_data in G.nodes(data=True):
        label = node_data.get("text", node_id)
        pos = node_data.get("upos", "unk")

        color = "lightblue"
        size = 10

        if positions:
            x, y = positions[node_id]
            net.add_node(
                node_id,
                label=label,
                x=x * 200, y=y * 200,
                color=color,
                size=size,
                title=pos
            )
        else:
            net.add_node(
                node_id,
                label=label,
                color=color,
                size=size,
                title=pos
            )

    # Add edges
    for u, v, edge_data in G.edges(data=True):
        label = edge_data.get("label", "")
        net.add_edge(u, v, label=label, title=label)

    logger.info(f"Saving Pyvis graph to {output_html}")
    net.show(output_html, notebook=False)

def generate_graph_summary(G: nx.MultiDiGraph) -> pd.DataFrame:
    """
    Generate a summary table of graph properties and grammatical features.
    """
    # Graph-level properties
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    num_entities = sum(1 for _, data in G.nodes(data=True) if data.get('entity', False))
    num_tokens = num_nodes - num_entities
    avg_degree = sum(dict(G.degree()).values()) / num_nodes if num_nodes > 0 else 0

    # Part-of-speech distribution
    pos_tags = [data['upos'] for _, data in G.nodes(data=True) if 'upos' in data]
    pos_counts = dict(Counter(pos_tags))

    # Dependency relations distribution
    #print(G.edges(data=True))
    dep_rels = [data['label'] for _, _, data in G.edges(data=True) if 'label' in data]
    dep_counts = dict(Counter(dep_rels))

    # Create a summary dictionary
    summary = {
        'Metric': ['Number of Nodes', 'Number of Edges', 'Number of Tokens', 'Number of Entities', 'Average Degree'],
        'Value': [num_nodes, num_edges, num_tokens, num_entities, avg_degree]
    }

    # Convert to DataFrame
    df_summary = pd.DataFrame(summary)

    # Add POS and dependency relation counts to the summary
    df_pos = pd.DataFrame(list(pos_counts.items()), columns=['POS Tag', 'Count'])
    df_pos = df_pos.sort_values(by='Count', ascending=False).reset_index(drop=True)
    df_pos['Percentage'] = (df_pos['Count'] / num_tokens * 100).round(2)

    df_dep = pd.DataFrame(list(dep_counts.items()), columns=['Dependency Relation', 'Count'])
    df_dep = df_dep.sort_values(by='Count', ascending=False).reset_index(drop=True)
    df_dep['Percentage'] = (df_dep['Count'] / num_edges * 100).round(2)

    return df_summary, df_pos, df_dep

# -----------------------------
# Example / main entrypoint
# -----------------------------

def main():
    # sample_text = (
    #     "Embora Maria estivesse trabalhando até tarde no escritório, correu para casa assim que recebeu "
    #     "uma ligação do irmão mais novo, que afirmava que o cachorro deles estava latindo furiosamente "
    #     "para algo no quintal. Quando chegou, no entanto, a única coisa que encontrou foi uma lixeira virada, "
    #     "o que sugeria que um guaxinim — ou talvez um gato de rua — tinha causado a confusão. "
    #     "Aliviada, mas ainda curiosa, Maria decidiu instalar uma pequena câmera perto da cerca "
    #     "para finalmente descobrir o que vinha perturbando o cachorro todas as noites."
    # )
    sample_text = open("../data/datasets/STF_HC/full/raw/Preso/1.txt").read().strip()
    G = build_document_graph(sample_text)

    df_summary, df_pos, df_dep = generate_graph_summary(G)

    # Display the summary tables
    print("Graph Summary:")
    print(df_summary)
    print("\nPart-of-Speech Distribution:")
    print(df_pos)
    print("\nDependency Relations Distribution:")
    print(df_dep)
    visualize_graph_pyvis(G, output_html="pt_doc_graph.html")

if __name__ == "__main__":
    main()


2025-10-01 12:54:08,548 [INFO] Loading spaCy Portuguese model pt_core_news_lg...
/home/trdp/Software/anaconda3/envs/phd_env/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'pt_core_news_lg' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.0). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Graph Summary:
               Metric      Value
0     Number of Nodes  758.00000
1     Number of Edges  792.00000
2    Number of Tokens  758.00000
3  Number of Entities    0.00000
4      Average Degree    2.08971

Part-of-Speech Distribution:
   POS Tag  Count  Percentage
0     NOUN    159       20.98
1      ADP    112       14.78
2    PUNCT    102       13.46
3    PROPN     77       10.16
4    SPACE     77       10.16
5     VERB     56        7.39
6      NUM     39        5.15
7      ADJ     38        5.01
8      DET     36        4.75
9    CCONJ     18        2.37
10    PRON     13        1.72
11   SCONJ     12        1.58
12     AUX     11        1.45
13     ADV      7        0.92
14       X      1        0.13

Dependency Relations Distribution:
   Dependency Relation  Count  Percentage
0                 case    110       13.89
1                punct    106       13.38
2                  dep     82       10.35
3                 nmod     71        8.96
4                  obl     39  

2025-10-01 12:54:12,876 [INFO] Saving Pyvis graph to pt_doc_graph.html


pt_doc_graph.html
